# Capítulo 7 · Eigensolver Variacional Cuántico (VQE)

## Objetivos

1. Comprender el principio variacional cuántico y su implementación en hardware ruidoso.
2. Construir un ansatz paramétrico (RealAmplitudes / EfficientSU2).
3. Calcular el valor esperado de un Hamiltoniano de Pauli con Qiskit.
4. Optimizar los parámetros con SPSA y COBYLA para encontrar la energía del estado fundamental.

---

## 7.1 Principio variacional

Para cualquier estado $|\psi(\boldsymbol{\theta})\rangle$ y Hamiltoniano $H$:

$$E_0 \leq \langle\psi(\boldsymbol{\theta})|H|\psi(\boldsymbol{\theta})\rangle \equiv E(\boldsymbol{\theta})$$

VQE minimiza $E(\boldsymbol{\theta})$ sobre el espacio paramétrico $\boldsymbol{\theta} \in \mathbb{R}^m$ mediante un optimizador clásico, obteniendo una aproximación al estado fundamental $E_0$.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', '..'))

import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.circuit.library import RealAmplitudes, EfficientSU2
from qiskit_aer import AerSimulator
from qiskit_aer.primitives import EstimatorV2 as Estimator
from scipy.optimize import minimize

print('Qiskit y dependencias cargados.')

## 7.2 Hamiltoniano de prueba: H₂ simplificado

Usaremos el Hamiltoniano de Heisenberg de 2 qubits como ejemplo pedagógico:

$$H = -J(X \otimes X + Y \otimes Y + Z \otimes Z)$$

con $J = 1$, cuyo estado fundamental es uno de los estados de Bell singlete $|\Psi^-\rangle$ con energía $E_0 = -3$.

In [ ]:
# Hamiltoniano de Heisenberg 2 qubits
J = 1.0
hamiltonian = SparsePauliOp.from_list([
    ('XX', -J),
    ('YY', -J),
    ('ZZ', -J),
])

# Energía exacta por diagonalización
H_matrix = hamiltonian.to_matrix()
eigenvalues = np.linalg.eigvalsh(H_matrix)
E_exact = np.min(eigenvalues)

print('Hamiltoniano de Heisenberg (2 qubits):')
print(hamiltonian)
print(f'\nEigenvalores: {eigenvalues}')
print(f'Energía del estado fundamental E₀ = {E_exact:.6f}')

## 7.3 Ansatz y evaluación del valor esperado

In [ ]:
# Ansatz: RealAmplitudes con 2 qubits y 2 capas
n_qubits = 2
n_reps   = 2
ansatz = RealAmplitudes(n_qubits, reps=n_reps, entanglement='linear')
n_params = ansatz.num_parameters

print(f'Ansatz: RealAmplitudes (n_qubits={n_qubits}, reps={n_reps})')
print(f'Número de parámetros: {n_params}')
print(ansatz.decompose().draw('text'))

# Estimator para evaluar ⟨ψ(θ)|H|ψ(θ)⟩
estimator = Estimator()

## 7.4 Bucle de optimización VQE

In [ ]:
energy_history = []

def cost_function(params: np.ndarray) -> float:
    """Calcula el valor esperado ⟨H⟩ para los parámetros dados."""
    bound_circuit = ansatz.assign_parameters(params)
    sv = Statevector(bound_circuit)
    H_matrix = hamiltonian.to_matrix()
    energy = np.real(sv.data.conj() @ H_matrix @ sv.data)
    energy_history.append(energy)
    return energy

# Parámetros iniciales aleatorios
np.random.seed(42)
theta_init = np.random.uniform(0, 2 * np.pi, n_params)

print(f'Energía inicial: {cost_function(theta_init):.6f}')
print(f'Energía exacta:  {E_exact:.6f}')
print('\nOptimizando con COBYLA...')

result = minimize(
    cost_function,
    theta_init,
    method='COBYLA',
    options={'maxiter': 500, 'rhobeg': 0.5},
)

E_vqe   = result.fun
theta_opt = result.x

print(f'\n=== Resultado VQE ===')
print(f'Energía VQE     = {E_vqe:.6f}')
print(f'Energía exacta  = {E_exact:.6f}')
print(f'Error absoluto  = {abs(E_vqe - E_exact):.2e}')
print(f'Iteraciones     = {result.nfev}')

In [ ]:
# Curva de convergencia
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(energy_history, color='#58a6ff', linewidth=1.5, label='E(θ) VQE')
ax.axhline(E_exact, color='#f78166', linestyle='--', linewidth=1.5,
           label=f'E₀ exacta = {E_exact:.4f}')
ax.set_xlabel('Iteración del optimizador')
ax.set_ylabel('Energía ⟨H⟩')
ax.set_title('Convergencia de VQE — Hamiltoniano de Heisenberg 2 qubits')
ax.legend()
ax.grid(alpha=0.3)
ax.set_facecolor('#161b22')
fig.patch.set_facecolor('#0d1117')
plt.tight_layout()
plt.show()

## 7.5 Verificación del estado óptimo

In [ ]:
# Estado óptimo
optimal_circuit = ansatz.assign_parameters(theta_opt)
sv_opt = Statevector(optimal_circuit)

print('Estado fundamental aproximado por VQE:')
for i, amp in enumerate(sv_opt.data):
    print(f'  |{format(i, "02b")}〉: {amp:.4f}  (prob = {np.abs(amp)**2:.4f})')

print('\nEstado singlete exacto |Ψ-〉 = (|01〉 - |10〉)/√2:')
psi_minus_exact = np.array([0, 1/np.sqrt(2), -1/np.sqrt(2), 0])
for i, amp in enumerate(psi_minus_exact):
    print(f'  |{format(i, "02b")}〉: {amp:.4f}  (prob = {np.abs(amp)**2:.4f})')

fidelidad = np.abs(np.dot(sv_opt.data.conj(), psi_minus_exact))**2
print(f'\nFidelidad VQE vs |Ψ-〉 = {fidelidad:.6f}')

## 7.6 Ejercicios propuestos

1. Repite el VQE con el optimizador SPSA, que no requiere gradiente. Compara el número de evaluaciones de circuito.

2. Añade ruido al simulador (modelo de decoherencia) y analiza cómo afecta a la energía mínima que el VQE puede alcanzar.

3. Prueba el ansatz `EfficientSU2` con parámetros complejos. ¿Necesitas más o menos iteraciones para converger?

4. Implementa VQE para el Hamiltoniano de Ising 1D de 4 qubits: $H = -J\sum_i Z_i Z_{i+1} - h\sum_i X_i$.